# 02 - DATA PREPROCESSING

**Mục tiêu:** Biến đổi dữ liệu thô từ PostgreSQL thành dữ liệu sạch đảm bảo tính nhất quán, hợp lệ và sẵn sàng cho phân tích khám phá dữ liệu (EDA) và mô hình hóa.

## Mục lục
- [I. Khởi tạo & Kết nối dữ liệu](#i-khởi-tạo-và-kết-nối-dữ-liệu)
- [II. Làm sạch & Tiền xử lý dữ liệu](#ii-tiền-xử-lý-các-bảng)
    - [1. Bảng Category](#1-bảng-category)
    - [2. Bảng Seller](#2-bảng-seller)
    - [3. Bảng Product](#3-bảng-product)
    - [4. Bảng Price Offer](#4-bảng-price-offer)
    - [5. Bảng Review](#5-bảng-review)
    - [6. Bảng Reviewer](#6-bảng-reviewer)
    - [7. Bảng Service](#7-bảng-service)
    - [8. Bảng Coupon](#8-bảng-coupon)
    - [3. Bảng Offer Coupon](#9-bảng-offer-coupon)
    - [3. Bảng Offer Service](#10-bảng-offer-service)
- [III. Lưu trữ & Xuất dữ liệu kết quả](#iii-lưu-trữ-dữ-liệu)

In [1]:
import sys
import os

root_path = os.path.abspath(os.path.join('..'))
if root_path not in sys.path:
    sys.path.append(root_path)

import polars as pl
from sqlalchemy import create_engine
from dotenv import load_dotenv

import warnings
warnings.filterwarnings('ignore')

## I. KHỞI TẠO VÀ KẾT NỐI DỮ LIỆU

Sử dụng `SQLAlchemy` để kết nối cơ sở dữ liệu và `dotenv` để bảo mật thông tin tài khoản. Dữ liệu được tải thông qua thư viện `Polars` nhằm tối ưu hiệu suất đọc và xử lý theo hướng đa luồng.

### Cấu hình kết nối
Các biến môi trường như `LOCAL_DB_USER`, `LOCAL_DB_PASS`, `LOCAL_DB_HOST`, `LOCAL_DB_NAME` và `LOCAL_DB_PORT` được nạp từ file môi trường để tạo chuỗi kết nối PostgreSQL một cách linh hoạt và an toàn.

In [2]:
load_dotenv()

USER = os.getenv("LOCAL_DB_USER")
PASSWORD = os.getenv("LOCAL_DB_PASS")
HOST = os.getenv("LOCAL_DB_HOST")
DBNAME = os.getenv("LOCAL_DB_NAME")
PORT = os.getenv("LOCAL_DB_PORT")

DATABASE_URL = f"postgresql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DBNAME}"
engine = create_engine(DATABASE_URL)

### Tải dữ liệu từ PostgreSQL
Sau khi khởi tạo `engine`, toàn bộ 10 bảng dữ liệu được truy vấn trực tiếp từ PostgreSQL bằng `pl.read_database(...)`. Việc tách dữ liệu theo từng thực thể giúp quá trình làm sạch dễ kiểm soát hơn trước khi đồng bộ quan hệ giữa các bảng.

In [3]:
with engine.connect() as conn:
    df_category = pl.read_database("SELECT * FROM category", conn)
    df_seller = pl.read_database("SELECT * FROM seller", conn)
    df_product = pl.read_database("SELECT * FROM product", conn)
    df_price_offer = pl.read_database("SELECT * FROM price_offer", conn)
    df_review = pl.read_database("SELECT * FROM review", conn)
    df_reviewer = pl.read_database("SELECT * FROM reviewer", conn)
    df_service = pl.read_database("SELECT * FROM service", conn)
    df_coupon = pl.read_database("SELECT * FROM coupon", conn)
    df_offer_coupon = pl.read_database("SELECT * FROM offer_coupon", conn)
    df_offer_service = pl.read_database("SELECT * FROM offer_service", conn)

## II. TIỀN XỬ LÝ CÁC BẢNG

### 1. Bảng Category

In [4]:
print(f"Kích thước bảng Category: {df_category.shape}")
print(f"Kiểu dữ liệu của bảng Category:\n{df_category.dtypes}")

print("Kiểm tra số lượng dữ liệu null trong bảng Category:")
display(df_category.null_count())

display(df_category.head())

Kích thước bảng Category: (3901, 7)
Kiểu dữ liệu của bảng Category:
[String, String, String, Int64, String, String, Boolean]
Kiểm tra số lượng dữ liệu null trong bảng Category:


category_id,category_name,parent_category_id,level,category_path,category_url,is_scanned
u32,u32,u32,u32,u32,u32,u32
0,0,26,0,0,0,0


category_id,category_name,parent_category_id,level,category_path,category_url,is_scanned
str,str,str,i64,str,str,bool
"""1789""","""Điện Thoại - Máy Tính Bảng""",null,1,"""Điện Thoại - Máy Tính Bảng""","""https://tiki.vn/dien-thoai-may…",true
"""1686""","""Giày - Dép nam""",null,1,"""Giày - Dép nam""","""https://tiki.vn/giay-dep-nam/c…",true
"""8322""","""Nhà Sách Tiki""",null,1,"""Nhà Sách Tiki""","""https://tiki.vn/nha-sach-tiki/…",true
"""4221""","""Điện Tử - Điện Lạnh""",null,1,"""Điện Tử - Điện Lạnh""","""https://tiki.vn/dien-tu-dien-l…",true
"""8371""","""Đồng hồ và Trang sức""",null,1,"""Đồng hồ và Trang sức""","""https://tiki.vn/dong-ho-va-tra…",true


In [5]:
df_category = df_category.unique(subset=["category_id"], keep='first') \
            .drop('is_scanned') \
            .with_columns([
                pl.col("category_id").cast(pl.Int64),
                pl.col("parent_category_id").cast(pl.Int64),
                pl.col("level").cast(pl.Int8),
                pl.col("category_path").str.strip_chars(),
                pl.col("category_name").str.strip_chars(),
                pl.col("category_url").str.strip_chars()
            ]) \
            .with_columns([
                pl.col('parent_category_id').fill_null(0)
            ]) \
            .sort(by=['level', 'category_id'])

### 2. Bảng Seller

Với bảng `Seller`, dữ liệu được làm sạch bằng cách loại bỏ trùng lặp theo `seller_id`, chuẩn hóa chuỗi ở `seller_name`, điền giá trị mặc định cho các trường còn thiếu và tối ưu kiểu dữ liệu số để giảm chi phí lưu trữ.

In [6]:
print(f"Kích thước bảng Seller: {df_seller.shape}")
print(f"Kiểu dữ liệu của bảng Seller:\n{df_seller.dtypes}")

print("Kiểm tra số lượng dữ liệu null trong bảng Seller:")
display(df_seller.null_count())
display(df_seller.head())

Kích thước bảng Seller: (4705, 4)
Kiểu dữ liệu của bảng Seller:
[String, String, Float64, Int64]
Kiểm tra số lượng dữ liệu null trong bảng Seller:


seller_id,seller_name,seller_rating,total_reviews
u32,u32,u32,u32
0,0,0,0


seller_id,seller_name,seller_rating,total_reviews
str,str,f64,i64
"""sach-tieng-nhat-jlpt""","""SÁCH TIẾNG NHẬT JLPT""",5.0,1
"""noi-that-nha-minh""","""NỘI THẤT ADORA NEMADORA""",4.8,32
"""bookcity-vn""","""BOOKCITY.VN""",4.8,1200
"""espp""","""ESPP""",4.5,117
"""2hbooks""","""2HBooks""",4.9,168


In [7]:
df_seller = df_seller.unique(subset=["seller_id"], keep="first")\
        .with_columns([
            pl.col("seller_id").str.strip_chars(),
            pl.col("seller_name").str.strip_chars()
        ])\
        .with_columns([
            pl.col("seller_rating").cast(pl.Float32),
            pl.col("total_reviews").cast(pl.Int32)
        ])\
        .with_columns([
            pl.col("seller_name").fill_null("Đang cập nhật"),
            pl.col("seller_rating").fill_null(0.0),
            pl.col("total_reviews").fill_null(0)
        ])

### 3. Bảng Product

Ngoài việc loại bỏ trùng lặp theo `product_id`, notebook còn chuẩn hóa tên sản phẩm, mô tả ngắn, thương hiệu/tác giả và các trường số liệu tổng hợp để giảm sai lệch trong các bước EDA sau này.

In [8]:
print(f"Kích thước bảng Product: {df_product.shape}")
print(f"Kiểu dữ liệu của bảng Product:\n{df_product.dtypes}")

print("Kiểm tra số lượng dữ liệu null trong bảng Product:")
display(df_product.null_count())
display(df_product.head())

Kích thước bảng Product: (310110, 11)
Kiểu dữ liệu của bảng Product:
[String, String, String, String, String, String, String, String, Int64, Int64, Float64]
Kiểm tra số lượng dữ liệu null trong bảng Product:


product_id,product_name,short_description,category_id,seller_id,product_url,image_url,author_brand,sold_quantity,review_count,review_score
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,134682,0,17,0,0,2526,0,0,0


product_id,product_name,short_description,category_id,seller_id,product_url,image_url,author_brand,sold_quantity,review_count,review_score
str,str,str,str,str,str,str,str,i64,i64,f64
"""275822950""","""Ốp lưng dẻo trong dành cho Hua…",null,"""28584""","""atzstore""","""https://tiki.vn/op-lung-deo-tr…","""https://salt.tikicdn.com/cache…","""GREENCASE""",0,0,0.0
"""273982699""","""Ốp lưng dẻo trong dành cho Hua…",null,"""28584""","""phu-kien-bingo""","""https://tiki.vn/op-lung-deo-tr…","""https://salt.tikicdn.com/cache…","""ULTRA THIN""",0,0,0.0
"""275236637""","""Luyện Thi Năng Lực Nhật Ngữ Tr…","""N/A""","""67878""","""tien-tho-book-hn""","""https://tiki.vn/luyen-thi-nang…","""https://salt.tikicdn.com/cache…","""""",0,0,0.0
"""273454695""","""2500 Từ Vựng Cần Thiết Cho Kỳ …","""N/A""","""67878""","""info-book""","""https://tiki.vn/2500-tu-vung-c…","""https://salt.tikicdn.com/cache…","""ARC ACADEMY""",0,0,0.0
"""273423214""","""Giáo Trình Luyện Thi Năng Lực …","""N/A""","""67878""","""nhan-van""","""https://tiki.vn/giao-trinh-luy…","""https://salt.tikicdn.com/cache…","""NHÓM TÁC GIẢ""",6,0,0.0


In [9]:
promo_pattern = r"(?i)(mua\s*\d|giảm\s*\d|giảm\s*k|giảm\s*%|tặng|freeship|hoàn tiền|voucher|đơn từ|rẻ hơn|sale|combo|ưu đãi|giá gốc)"

df_product = df_product.unique(subset=["product_id"], keep="first")\
            .with_columns([
                pl.col("product_id").cast(pl.Int64),
                pl.col("category_id").cast(pl.Int64),
                pl.col("seller_id").str.strip_chars(),
                pl.col("product_name").str.strip_chars(),
                pl.col("author_brand").str.strip_chars(),
                pl.col("short_description").str.strip_chars().replace("N/A", None),
                pl.col("image_url").str.strip_chars()
            ])\
            .with_columns(
                pl.when(pl.col("author_brand").str.contains(promo_pattern))
                .then(pl.lit(None))
                .otherwise(pl.col("author_brand"))
                .alias("author_brand")
            )\
            .with_columns(
                pl.col("author_brand")
                .str.replace(r"(?i)^(0|null|n/a|na|none)$", "")
                .replace("", None)
            )\
            .with_columns([
                pl.col("sold_quantity").cast(pl.Int32).fill_null(0),
                pl.col("review_count").cast(pl.Int32).fill_null(0),
                pl.col("review_score").cast(pl.Float32).fill_null(0.0)
            ])

df_product.head()

product_id,product_name,short_description,category_id,seller_id,product_url,image_url,author_brand,sold_quantity,review_count,review_score
i64,str,str,i64,str,str,str,str,i32,i32,f32
277772657,"""Sách - Phụng Nhãn - Bản Thường…",null,67936,"""bamboo-books""","""https://tiki.vn/sach-phung-nha…","""https://salt.tikicdn.com/cache…",null,0,0,0.0
95280506,"""Viên đá Ruby thiên nhiên_HA-RB…",null,68434,"""kim-cuong-da-quy-hai-anh""","""https://tiki.vn/vien-da-ruby-t…","""https://salt.tikicdn.com/cache…","""OEM""",0,0,0.0
248846486,"""TRÚC THƯ DAO 2 – NƯỚC TẤN – GI…",null,67996,"""binh-ban-book""","""https://tiki.vn/truc-thu-dao-2…","""https://salt.tikicdn.com/cache…",null,1,0,0.0
276105014,"""Bình đun siêu tốc Braun WK1500…",null,1931,"""long-hung-official-store""","""https://tiki.vn/binh-dun-sieu-…","""https://salt.tikicdn.com/cache…","""BRAUN""",1,0,0.0
274862964,"""Sách - Illustrated Classics - …",null,1753,"""nhung-vi-sao-bookstore""","""https://tiki.vn/sach-illustrat…","""https://salt.tikicdn.com/cache…","""NHIEU TAC GIA""",0,0,0.0


### 4. Bảng Price Offer

Bảng `Price Offer` ghi nhận thông tin giá theo thời điểm crawl. Quá trình làm sạch tập trung vào loại bỏ bản ghi trùng, chuẩn hóa định danh `offer_id`, `product_id`, kiểm soát giá trị rỗng và ép kiểu các trường giá về dạng số thực để sẵn sàng cho phân tích biến động giá.

In [10]:
print(f"Kích thước bảng Price_Offer: {df_price_offer.shape}")
print(f"Kiểu dữ liệu của bảng Price_Offer:\n{df_price_offer.dtypes}")

print("Kiểm tra số lượng dữ liệu null trong bảng Price_Offer:")
display(df_price_offer.null_count())
display(df_price_offer.head())

Kích thước bảng Price_Offer: (953445, 6)
Kiểu dữ liệu của bảng Price_Offer:
[String, String, Float64, Float64, Float64, Datetime(time_unit='us', time_zone=None)]
Kiểm tra số lượng dữ liệu null trong bảng Price_Offer:


offer_id,product_id,current_price,original_price,discount_percent,crawl_time
u32,u32,u32,u32,u32,u32
0,0,0,0,0,0


offer_id,product_id,current_price,original_price,discount_percent,crawl_time
str,str,f64,f64,f64,datetime[μs]
"""633304a834""","""1217527""",48000.0,48000.0,0.0,2026-02-25 19:20:12.984106
"""7d57439ca7""","""1217917""",80000.0,80000.0,0.0,2026-02-25 19:20:16.744204
"""ca2719fc3b""","""1217919""",80000.0,80000.0,0.0,2026-02-25 19:20:22.292313
"""7a2a3f51a4""","""1217923""",142000.0,142000.0,0.0,2026-02-25 19:20:28.327105
"""52890ddf91""","""1217929""",69900.0,69900.0,0.0,2026-02-25 19:20:31.262106


In [11]:
df_price_offer = df_price_offer.unique(subset=["offer_id"], keep='first') \
    .with_columns([
        pl.col("offer_id").str.strip_chars(),
        pl.col("product_id").str.strip_chars()
    ]) \
    .filter(
        pl.col("offer_id").is_not_null() & (pl.col("offer_id") != "") & 
        pl.col("product_id").is_not_null() & (pl.col("product_id") != "")
    ) \
    .with_columns([
        pl.col("current_price").cast(pl.Float64).fill_null(0.0),
        pl.col("original_price").cast(pl.Float64).fill_null(0.0),
        pl.col("discount_percent").cast(pl.Float64).fill_null(0.0)
    ])

print(f"Kích thước bảng Price_Offer sau khi làm sạch: {df_price_offer.shape}")
display(df_price_offer.head())

Kích thước bảng Price_Offer sau khi làm sạch: (953445, 6)


offer_id,product_id,current_price,original_price,discount_percent,crawl_time
str,str,f64,f64,f64,datetime[μs]
"""a0cf453a55""","""120169054""",89000.0,89000.0,0.0,2026-02-24 23:58:11.279590
"""a4b77d002b""","""279038348""",185200.0,195000.0,5.0,2026-03-05 14:19:34.245679
"""f20a9c44b8""","""178478019""",7.322e6,7.322e6,0.0,2026-03-04 21:31:24.069740
"""5bc8b2ac41""","""118235667""",385000.0,385000.0,0.0,2026-03-13 09:59:59.249588
"""c822813268""","""117891921""",35000.0,55000.0,36.0,2026-03-03 13:57:50.770479


### 5. Bảng Review

Bảng `Review` được chuẩn hóa cả về nội dung văn bản lẫn ý nghĩa thời gian. Các biểu thức Regex được sử dụng để tách số lượng thời gian từ các chuỗi như `Đánh giá vào 2 năm trước` hoặc `Đã dùng 3 tháng`, sau đó quy đổi về số ngày để thuận lợi cho phân tích hành vi người dùng.

In [12]:
print(f"Kích thước bảng Review: {df_review.shape}")
print(f"Kiểu dữ liệu của bảng Review:\n{df_review.dtypes}")

print("Kiểm tra số lượng dữ liệu null trong bảng Review:")
display(df_review.null_count())
display(df_review.head())

Kích thước bảng Review: (1320038, 8)
Kiểu dữ liệu của bảng Review:
[String, String, String, Int64, String, Int64, String, String]
Kiểm tra số lượng dữ liệu null trong bảng Review:


review_id,product_id,reviewer_id,rating_score,review_content,thank_count,review_time,usage_duration
u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,165379,0,0,4714


review_id,product_id,reviewer_id,rating_score,review_content,thank_count,review_time,usage_duration
str,str,str,i64,str,i64,str,str
"""d2f16143-fe9""","""273377046""","""679284b1e4""",5,"""""",0,"""Đánh giá vào 5 tháng trước""","""Đã dùng 2 ngày"""
"""3105eeae-88b""","""262803182""","""c9a9bab8f4""",4,"""""",0,"""Đánh giá vào 2 năm trước""","""Đã dùng 3 tháng"""
"""f0ea86b7-f9b""","""163739509""","""36138811b4""",5,"""""",0,"""Đánh giá vào 2 năm trước""","""Đã dùng 4 tháng"""
"""6cbb0f56-49c""","""163739509""","""70b76f17ab""",5,"""sách tốt, như ảnh""",0,"""Đánh giá vào 4 năm trước""","""Đã dùng 3 ngày"""
"""7446cd0a-d0e""","""163739509""","""574ea2859b""",5,"""""",0,"""Đánh giá vào 3 năm trước""","""Đã dùng 4 tháng"""


In [13]:
df_review = (
    df_review.unique(subset=["review_id"], keep='first')
    .with_columns([
        pl.col("rating_score").cast(pl.Int8),
        pl.col("thank_count").cast(pl.Int32).fill_null(0),
        
        pl.col("review_content").str.replace_all(r"\s+", " ").str.strip_chars().fill_null(""),
        
        pl.col("review_time").str.extract(r"(\d+)").cast(pl.Int32).fill_null(0).alias("time_num"),
        pl.col("review_time").str.extract(r"(năm|tháng|tuần|ngày|giờ|phút)").str.to_lowercase().fill_null("ngày").alias("time_unit"),
        
        pl.col("usage_duration").str.extract(r"(\d+)").cast(pl.Int32).fill_null(0).alias("usage_num"),
        pl.col("usage_duration").str.extract(r"(năm|tháng|tuần|ngày)").str.to_lowercase().fill_null("ngày").alias("usage_unit"),
    ])
    .with_columns([
        (
            pl.col("time_num") * pl.when(pl.col("time_unit") == "năm").then(365)
            .when(pl.col("time_unit") == "tháng").then(30)
            .when(pl.col("time_unit") == "tuần").then(7)
            .when(pl.col("time_unit").is_in(["giờ", "phút"])).then(0)
            .otherwise(1)
        ).alias("review_days_ago"),
        
        (
            pl.col("usage_num") * pl.when(pl.col("usage_unit") == "năm").then(365)
            .when(pl.col("usage_unit") == "tháng").then(30)
            .when(pl.col("usage_unit") == "tuần").then(7)
            .otherwise(1)
        ).alias("usage_days")
    ])
    .drop(["review_time", "usage_duration", "time_num", "time_unit", "usage_num", "usage_unit"])
    .sort(by=["product_id", "review_id"])
)

display(df_review.head())

review_id,product_id,reviewer_id,rating_score,review_content,thank_count,review_days_ago,usage_days
str,str,str,i8,str,i32,i32,i32
"""1420ac0e-273""","""100023508""","""96d51f72e6""",5,"""""",0,365,2
"""359f05e9-b36""","""100023508""","""6eaff986b6""",5,"""Ok""",0,1460,8
"""672816e1-feb""","""100023508""","""6ce1f186bc""",5,"""""",0,1460,11
"""71aaffae-baf""","""100023508""","""75ff41909c""",5,"""""",0,1460,3
"""7bcf5698-ec6""","""100023508""","""1e8c004cc3""",5,"""""",0,1460,5


### 6. Bảng Reviewer

Bảng `Reviewer` được làm sạch bằng cách chuẩn hóa tên người dùng, xử lý giá trị thiếu và chuyển thông tin thâm niên từ dạng mô tả ngôn ngữ tự nhiên sang số ngày (`seniority_days`) để tăng khả năng định lượng trong phân tích.

In [14]:
print(f"Kích thước bảng Reviewer: {df_reviewer.shape}")
print(f"Kiểu dữ liệu của bảng Reviewer:\n{df_reviewer.dtypes}")

print("Kiểm tra số lượng dữ liệu null trong bảng Reviewer:")
display(df_reviewer.null_count())
display(df_reviewer.head())

Kích thước bảng Reviewer: (458141, 5)
Kiểu dữ liệu của bảng Reviewer:
[String, String, String, Int64, Int64]
Kiểm tra số lượng dữ liệu null trong bảng Reviewer:


reviewer_id,reviewer_name,reviewer_seniority,reviewer_contributions,reviewer_received_thanks
u32,u32,u32,u32,u32
0,2,2,0,0


reviewer_id,reviewer_name,reviewer_seniority,reviewer_contributions,reviewer_received_thanks
str,str,str,i64,i64
"""002304bb1b""","""Đào Quốc Thiện""","""Đã tham gia 7 năm""",21,3
"""ab814822eb""","""Quang Tuệ""","""Đã tham gia 8 năm""",64,0
"""457ec04452""","""Luân Trịnh""","""Đã tham gia 4 năm""",3,0
"""0b4eb78dff""","""Tất Liên""","""Đã tham gia 9 năm""",52,0
"""66407913dc""","""Huệ Nguyễn""","""Đã tham gia 12 năm""",1,0


In [15]:
df_reviewer = (
    df_reviewer.unique(subset=["reviewer_id"], keep='first')
    .with_columns([
        pl.col("reviewer_id").str.strip_chars(),
        pl.col("reviewer_name").str.strip_chars().fill_null("Ẩn danh"),
        pl.col("reviewer_contributions").cast(pl.Int32).fill_null(0),
        pl.col("reviewer_received_thanks").cast(pl.Int32).fill_null(0),
        
        pl.col("reviewer_seniority").str.extract(r"(\d+)").cast(pl.Int32).fill_null(0).alias("seniority_num"),
        pl.col("reviewer_seniority").str.extract(r"(năm|tháng|tuần|ngày)").str.to_lowercase().fill_null("ngày").alias("seniority_unit")
    ])
    .with_columns([
        (
            pl.col("seniority_num") * pl.when(pl.col("seniority_unit") == "năm").then(365)
            .when(pl.col("seniority_unit") == "tháng").then(30)
            .when(pl.col("seniority_unit") == "tuần").then(7)
            .otherwise(1)
        ).alias("seniority_days")
    ])
    .drop(["reviewer_seniority", "seniority_num", "seniority_unit"])
    .sort(by=["reviewer_id"])
)

display(df_reviewer.head())

reviewer_id,reviewer_name,reviewer_contributions,reviewer_received_thanks,seniority_days
str,str,i32,i32,i32
"""000015492f""","""Rita Nguyen""",140,4,3650
"""000042f447""","""lã khánh linh""",6,1,1460
"""00008f75cb""","""Nguyễn Thị Bích Lệ""",7,2,3650
"""0000b7fbc0""","""lê thị hường""",23,2,1825
"""0000c9136b""","""Phan Ngọc Như""",32,23,2555


### 7. Bảng Service

Bảng `Service` lưu các dịch vụ đi kèm sản phẩm/đơn hàng. Việc tiền xử lý nhằm chuẩn hóa mã dịch vụ, loại bỏ bản ghi không hợp lệ và giữ cho tập dữ liệu nhất quán trước khi liên kết sang các bảng trung gian.

In [16]:
print(f"Kích thước bảng Service: {df_service.shape}")
print(f"Kiểu dữ liệu của bảng Service:\n{df_service.dtypes}")

print("Kiểm tra số lượng dữ liệu null trong bảng Service:")
display(df_service.null_count())
display(df_service.head())

Kích thước bảng Service: (1387, 2)
Kiểu dữ liệu của bảng Service:
[Int64, String]
Kiểm tra số lượng dữ liệu null trong bảng Service:


service_id,service_name
u32,u32
0,0


service_id,service_name
i64,str
249067,"""Trả góp từ 657.500 ₫/tháng"""
249081,"""Trả góp từ 833.250 ₫/tháng"""
249086,"""Trả góp từ 383.250 ₫/tháng"""
249071,"""Trả góp từ 415.833 ₫/tháng"""
249141,"""Trả góp từ 1.523.333 ₫/tháng"""


In [17]:
df_service = (
        df_service
        .unique(subset=["service_id"], keep="first")
        
        .with_columns([
            pl.col("service_id").cast(pl.Int64),
            pl.col("service_name").str.strip_chars()
        ])
        
        .with_columns([
            pl.col("service_name")
            .str.extract(r"(\d{1,3}(?:\.\d{3})+)", 1)
            .str.replace_all(r"\.", "")
            .cast(pl.Int64, strict=False)
            .alias("installment_price_vnd")
        ])
    )

display(df_service.head())

service_id,service_name,installment_price_vnd
i64,str,i64
249461,"""Trả góp từ 260.083 ₫/tháng""",260083
249857,"""Trả góp từ 891.000 ₫/tháng""",891000
250241,"""Trả góp từ 374.833 ₫/tháng""",374833
249976,"""Trả góp từ 582.750 ₫/tháng""",582750
302674,"""Trả góp từ 268.250 ₫/tháng""",268250


### 8. Bảng Coupon

Bảng `Coupon` được làm sạch theo hướng chuẩn hóa mã giảm giá, mô tả và các trường định danh để hỗ trợ phân tích hiệu quả khuyến mãi cũng như liên kết với bảng ưu đãi trung gian.

In [18]:
print(f"Kích thước bảng Coupon: {df_coupon.shape}")
print(f"Kiểu dữ liệu của bảng Coupon:\n{df_coupon.dtypes}")

print("Kiểm tra số lượng dữ liệu null trong bảng Coupon:")
display(df_coupon.null_count())
display(df_coupon.head())

Kích thước bảng Coupon: (1044, 5)
Kiểu dữ liệu của bảng Coupon:
[Int64, String, String, String, Date]
Kiểm tra số lượng dữ liệu null trong bảng Coupon:


coupon_id,coupon_code,title,condition,expiry
u32,u32,u32,u32,u32
0,0,0,0,0


coupon_id,coupon_code,title,condition,expiry
i64,str,str,str,date
4923,"""TET2026""","""Giảm 15K""","""Cho đơn hàng từ 300K""",2026-04-30
1933,"""DEHA53""","""Giảm 50%""","""Cho đơn hàng từ 50K""",2026-02-28
1707,"""10KY26""","""Giảm 10K""","""Cho đơn hàng từ 770K""",2026-02-28
22459,"""PATYCA5""","""Giảm 5K""","""Cho đơn hàng từ 100K""",2026-12-31
259926,"""FEB100KK""","""Giảm 100K""","""Cho đơn hàng từ 3 triệu""",2026-02-28


In [19]:
df_coupon = (
        df_coupon
        .unique(subset=["coupon_id"], keep="first")
        .with_columns([
            pl.col("coupon_id").cast(pl.Int64),
            pl.col("coupon_code").str.strip_chars().fill_null("NO_CODE"),
            pl.col("title").str.strip_chars(),
            pl.col("condition").str.strip_chars()
        ])
        .with_columns([
            pl.col("expiry").cast(pl.String).str.to_date("%Y-%m-%d", strict=False)
        ])
    )

display(df_coupon.head())

coupon_id,coupon_code,title,condition,expiry
i64,str,str,str,date
134,"""E2O8YZB1NX""","""Giảm 7K""","""Cho đơn hàng từ 599K""",2026-12-31
144321,"""FPTFZ91OM0""","""Giảm 5% tối đa 50K""","""Cho đơn hàng từ 500K""",2026-03-05
17182,"""4X111""","""Giảm 4% tối đa 10K""","""Cho đơn hàng từ 250K""",2026-02-28
271091,"""IND25K""","""Giảm 25K""","""Cho đơn hàng từ 649K""",2026-02-28
14160,"""DTPBTET10""","""Giảm 10%""","""Số lượng có hạn""",2026-02-28


### 9. Bảng Offer Coupon

Đây là bảng liên kết nhiều-nhiều giữa `Price Offer` và `Coupon`. Quá trình làm sạch tập trung vào loại bỏ trùng lặp theo cặp khóa, chuẩn hóa định danh và đảm bảo mỗi liên kết đều có ý nghĩa phân tích.

In [20]:
print(f"Kích thước bảng Offer_Coupon: {df_offer_coupon.shape}")
print(f"Kiểu dữ liệu của bảng Offer_Coupon:\n{df_offer_coupon.dtypes}")

print("Kiểm tra số lượng dữ liệu null trong bảng Offer_Coupon:")
display(df_offer_coupon.null_count())
display(df_offer_coupon.head())

Kích thước bảng Offer_Coupon: (1352435, 2)
Kiểu dữ liệu của bảng Offer_Coupon:
[String, Int64]
Kiểm tra số lượng dữ liệu null trong bảng Offer_Coupon:


offer_id,coupon_id
u32,u32
0,0


offer_id,coupon_id
str,i64
"""f4cb55c547""",1
"""f4cb55c547""",2
"""f4cb55c547""",3
"""f4cb55c547""",4
"""f4cb55c547""",5


In [21]:
df_offer_coupon = df_offer_coupon.unique(subset=["offer_id", "coupon_id"], keep='first') \
    .with_columns([
        pl.col("offer_id").str.strip_chars(),
        pl.col("coupon_id").cast(pl.Int64)
    ]) \
    .drop_nulls() \
    .filter(
        pl.col("offer_id") != ""
    )

print(f"Kích thước bảng Offer_Coupon sau khi làm sạch: {df_offer_coupon.shape}")
display(df_offer_coupon.head())

Kích thước bảng Offer_Coupon sau khi làm sạch: (1352435, 2)


offer_id,coupon_id
str,i64
"""d32b50613c""",12
"""1486b9c6b5""",11
"""2385c2c75f""",11
"""0012d79cce""",1
"""000920ed85""",139912


### 10. Bảng Offer Service

Tương tự `Offer Coupon`, bảng `Offer Service` đóng vai trò trung gian để biểu diễn các dịch vụ áp dụng cho từng offer. Dữ liệu được chuẩn hóa khóa và loại bỏ bản ghi rỗng trước khi phục vụ bước kiểm tra liên kết.

In [22]:
print(f"Kích thước bảng Offer_Service: {df_offer_service.shape}")
print(f"Kiểu dữ liệu của bảng Offer_Service:\n{df_offer_service.dtypes}")

print("Kiểm tra số lượng dữ liệu null trong bảng Offer_Service:")
display(df_offer_service.null_count())
display(df_offer_service.head())

Kích thước bảng Offer_Service: (827972, 2)
Kiểu dữ liệu của bảng Offer_Service:
[String, Int64]
Kiểm tra số lượng dữ liệu null trong bảng Offer_Service:


offer_id,service_id
u32,u32
0,0


offer_id,service_id
str,i64
"""282f24cadf""",1
"""282f24cadf""",2
"""f4cb55c547""",1
"""f4cb55c547""",2
"""8c55765cab""",1


In [23]:
df_offer_service = df_offer_service.unique(subset=["offer_id", "service_id"], keep='first') \
    .with_columns([
        pl.col("offer_id").str.strip_chars(),
        pl.col("service_id").cast(pl.Int64)
    ]) \
    .drop_nulls() \
    .filter(
        pl.col("offer_id") != ""
    )

print(f"Kích thước bảng Offer_Service sau khi làm sạch: {df_offer_service.shape}")
display(df_offer_service.head())

Kích thước bảng Offer_Service sau khi làm sạch: (827972, 2)


offer_id,service_id
str,i64
"""dbafc84bda""",1
"""e6147dc4b1""",2
"""93b681df6b""",1
"""22d24a7fbd""",2
"""5eeb726379""",1


## III. LƯU TRỮ DỮ LIỆU

Dữ liệu sau khi làm sạch được lưu dưới định dạng `.parquet` để phục vụ các bước phân tích tiếp theo.

**Lý do chọn Parquet:**
- Giữ nguyên schema và kiểu dữ liệu sau tiền xử lý.
- Dung lượng nén thường nhỏ hơn đáng kể so với `.csv`.
- Tốc độ đọc/ghi tối ưu hơn cho khối lượng dữ liệu lớn và các pipeline phân tích dữ liệu.

In [24]:
SAVE_PATH = "../data/processed"

df_category.write_parquet(f"{SAVE_PATH}/category.parquet")
df_seller.write_parquet(f"{SAVE_PATH}/seller.parquet")
df_product.write_parquet(f"{SAVE_PATH}/product.parquet")
df_price_offer.write_parquet(f"{SAVE_PATH}/price_offer.parquet")
df_review.write_parquet(f"{SAVE_PATH}/review.parquet")
df_reviewer.write_parquet(f"{SAVE_PATH}/reviewer.parquet")
df_service.write_parquet(f"{SAVE_PATH}/service.parquet")
df_coupon.write_parquet(f"{SAVE_PATH}/coupon.parquet")
df_offer_coupon.write_parquet(f"{SAVE_PATH}/offer_coupon.parquet")
df_offer_service.write_parquet(f"{SAVE_PATH}/offer_service.parquet")